# NSW EV Intelligence Platform - RAG Chat Application

This notebook implements a complete **Retrieval-Augmented Generation (RAG)** chat application for your NSW EV data.

## What This Does
1. **Retrieval Function** - Queries the vector search index to find relevant documents
2. **LLM Integration** - Uses Databricks Foundation Models (DBRX Instruct) for response generation
3. **Prompt Template** - Combines user questions with retrieved context
4. **Chat Interface** - Gradio-based interactive chat UI

## Architecture
```
User Question → Vector Search (Retrieve) → Context + Question → LLM (Generate) → Answer
```

## Data Source
- Vector Search Index: `mobility_ai.rag.ev_documents_index`
- Documents: 117 EV charging stations + route safety reports

In [0]:
%pip install databricks-vectorsearch databricks-sdk gradio openai --upgrade --quiet
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
import json

# Initialize Databricks client
w = WorkspaceClient()

# Configuration
INDEX_NAME = "mobility_ai.rag.ev_documents_index"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"  # Databricks Foundation Model
TOP_K = 5  # Number of documents to retrieve

print("✓ Configuration loaded")
print(f"  Vector Search Index: {INDEX_NAME}")
print(f"  LLM Endpoint: {LLM_ENDPOINT}")
print(f"  Top K Results: {TOP_K}")

## Step 1: Build Retrieval Function

The retrieval function queries the vector search index to find the most relevant documents based on semantic similarity to the user's question.

In [0]:
def retrieve_relevant_documents(query: str, top_k: int = TOP_K):
    """
    Retrieve relevant documents from the vector search index.
    
    Args:
        query: User's question or search query
        top_k: Number of top results to return
        
    Returns:
        List of tuples: [(doc_id, content, source_table, score), ...]
    """
    try:
        # Query the vector search index
        results = w.vector_search_indexes.query_index(
            index_name=INDEX_NAME,
            columns=["doc_id", "content", "source_table"],
            query_text=query,
            num_results=top_k
        )
        
        # Extract and format results
        documents = []
        if results.result and results.result.data_array:
            for row in results.result.data_array:
                doc_id = row[0]
                content = row[1]
                source_table = row[2]
                score = row[-1]  # Similarity score is last column
                
                documents.append({
                    "doc_id": doc_id,
                    "content": content,
                    "source_table": source_table,
                    "score": float(score)
                })
        
        return documents
    
    except Exception as e:
        print(f"Error retrieving documents: {e}")
        return []

# Test the retrieval function
test_query = "Find fast charging stations near Sydney"
print(f"Testing retrieval with query: '{test_query}'\n")

results = retrieve_relevant_documents(test_query, top_k=3)
if results:
    print(f"✓ Retrieved {len(results)} documents:\n")
    for i, doc in enumerate(results, 1):
        print(f"{i}. Score: {doc['score']:.4f}")
        print(f"   Content: {doc['content'][:150]}...\n")
else:
    print("⚠ No results returned. The index may not be ready yet.")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# List all serving endpoints
endpoints = w.serving_endpoints.list()

print("Available Serving Endpoints:")
print("=" * 80)

foundation_models = []
custom_endpoints = []

for endpoint in endpoints:
    endpoint_name = endpoint.name
    
    # Check if it's a foundation model (typically starts with 'databricks-')
    if endpoint_name.startswith('databricks-'):
        foundation_models.append(endpoint_name)
    else:
        custom_endpoints.append(endpoint_name)

if foundation_models:
    print("\n📚 Foundation Model Endpoints:")
    for name in sorted(foundation_models):
        print(f"  • {name}")
else:
    print("\n⚠️  No Foundation Model endpoints found")

if custom_endpoints:
    print(f"\n🔧 Custom Endpoints ({len(custom_endpoints)}):")
    for name in sorted(custom_endpoints)[:10]:  # Show first 10
        print(f"  • {name}")
    if len(custom_endpoints) > 10:
        print(f"  ... and {len(custom_endpoints) - 10} more")
        
print("\n" + "=" * 80)

## Step 2: LLM Integration

Integrate with Databricks Foundation Models API to generate responses. We'll use **DBRX Instruct**, a high-quality instruction-following model.

In [0]:
def generate_response(prompt: str, max_tokens: int = 500):
    """
    Generate a response using Databricks Foundation Models.
    
    Args:
        prompt: The complete prompt with context and question
        max_tokens: Maximum tokens in the response
        
    Returns:
        str: Generated response
    """
    try:
        # Create chat messages
        messages = [
            ChatMessage(
                role=ChatMessageRole.SYSTEM,
                content="You are a helpful assistant for the NSW EV Intelligence Platform. You provide accurate information about electric vehicle charging stations, route safety, and traffic conditions in New South Wales, Australia. Base your answers strictly on the provided context."
            ),
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt
            )
        ]
        
        # Call the LLM endpoint
        response = w.serving_endpoints.query(
            name="databricks-meta-llama-3-3-70b-instruct",
            messages=messages,
            max_tokens=max_tokens,
            temperature=0.3  # Lower temperature for more focused responses
        )
        
        # Extract the response text
        if response.choices and len(response.choices) > 0:
            return response.choices[0].message.content
        else:
            return "I'm sorry, I couldn't generate a response."
    
    except Exception as e:
        print(f"Error generating response: {e}")
        return f"Error: {str(e)}"

# Test the LLM function
test_prompt = "Hello! Can you help me find EV charging stations?"
print(f"Testing LLM with prompt: '{test_prompt}'\n")

response = generate_response(test_prompt)
print(f"Response: {response}")

## Step 3: Prompt Template

Create a prompt template that combines the user's question with retrieved context. This is crucial for RAG - the LLM needs both the question and relevant information to generate accurate answers.

In [0]:
def create_rag_prompt(question: str, documents: list) -> str:
    """
    Create a RAG prompt by combining user question with retrieved documents.
    
    Args:
        question: User's question
        documents: List of retrieved document dictionaries
        
    Returns:
        str: Formatted prompt for the LLM
    """
    if not documents:
        return f"""I don't have any relevant information to answer this question. Please rephrase or ask about:
- Electric vehicle charging stations in NSW
- Route safety and traffic hazards
- Charging station locations and specifications

Question: {question}"""
    
    # Build context from retrieved documents
    context_parts = []
    for i, doc in enumerate(documents, 1):
        context_parts.append(f"Document {i} (Relevance Score: {doc['score']:.3f}):\n{doc['content']}")
    
    context = "\n\n".join(context_parts)
    
    # Create the complete prompt
    prompt = f"""You are an AI assistant for the NSW EV Intelligence Platform. Answer the user's question based ONLY on the provided context documents. 

IMPORTANT GUIDELINES:
- Use only information from the context documents below
- If the context doesn't contain enough information, say so clearly
- Be specific and cite details from the documents (locations, operators, ratings, etc.)
- For charging stations, include: name, location, operator, charging type, and speed when available
- For route safety, include: safety rating, hazard types, and estimated delays when available
- Format your response in a clear, user-friendly way

CONTEXT DOCUMENTS:
{context}

---

USER QUESTION: {question}

ANSWER (based strictly on the context above):"""
    
    return prompt

# Test the prompt template
test_question = "Where can I find fast charging stations?"
test_docs = retrieve_relevant_documents(test_question, top_k=2)

if test_docs:
    prompt = create_rag_prompt(test_question, test_docs)
    print("Sample RAG Prompt:")
    print("=" * 80)
    print(prompt[:800] + "...\n[truncated]")
    print("=" * 80)
else:
    print("⚠ No documents retrieved for testing")

## Step 4: Complete RAG Pipeline

Combine retrieval, prompt creation, and generation into a single pipeline function.

In [0]:
def rag_pipeline(question: str, top_k: int = TOP_K, verbose: bool = False):
    """
    Complete RAG pipeline: Retrieve → Prompt → Generate.
    
    Args:
        question: User's question
        top_k: Number of documents to retrieve
        verbose: Print debug information
        
    Returns:
        dict: Contains 'answer', 'sources', and 'retrieved_docs'
    """
    if verbose:
        print(f"\n{'='*80}")
        print(f"RAG PIPELINE - Question: {question}")
        print(f"{'='*80}\n")
    
    # Step 1: Retrieve relevant documents
    if verbose:
        print(f"[1/3] Retrieving {top_k} relevant documents...")
    
    documents = retrieve_relevant_documents(question, top_k=top_k)
    
    if verbose:
        print(f"      Retrieved {len(documents)} documents")
        if documents:
            print(f"      Top score: {documents[0]['score']:.4f}")
    
    # Step 2: Create RAG prompt
    if verbose:
        print(f"\n[2/3] Creating RAG prompt...")
    
    prompt = create_rag_prompt(question, documents)
    
    if verbose:
        print(f"      Prompt length: {len(prompt)} characters")
    
    # Step 3: Generate response
    if verbose:
        print(f"\n[3/3] Generating response with {LLM_ENDPOINT}...")
    
    answer = generate_response(prompt)
    
    if verbose:
        print(f"      Response generated ({len(answer)} characters)")
        print(f"\n{'='*80}\n")
    
    # Format sources for citation
    sources = []
    for doc in documents:
        source_info = {
            "id": doc["doc_id"],
            "score": doc["score"],
            "preview": doc["content"][:200] + "..."
        }
        sources.append(source_info)
    
    return {
        "answer": answer,
        "sources": sources,
        "retrieved_docs": documents
    }

# Test the complete RAG pipeline
print("\n" + "="*80)
print("TESTING COMPLETE RAG PIPELINE")
print("="*80)

test_questions = [
    "Where can I find fast charging stations near Sydney?",
    "Are there any route hazards I should be aware of?"
]

for question in test_questions:
    print(f"\n\nQuestion: {question}")
    print("-" * 80)
    
    result = rag_pipeline(question, top_k=3, verbose=False)
    
    print(f"\nAnswer:\n{result['answer']}")
    
    if result['sources']:
        print(f"\n\nSources ({len(result['sources'])} documents):")
        for i, source in enumerate(result['sources'], 1):
            print(f"  {i}. {source['id']} (Score: {source['score']:.4f})")

## Step 5: Gradio Chat Interface

Create an interactive chat interface using Gradio. Users can ask questions and get AI-powered answers based on your NSW EV data.

In [0]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import datetime

# Chat history storage
chat_history = []

def format_message(role, content, timestamp):
    """Format a chat message with HTML styling."""
    if role == "user":
        bg_color = "#e3f2fd"
        align = "right"
        label = "You"
    else:
        bg_color = "#f5f5f5"
        align = "left"
        label = "🚗⚡ Assistant"
    
    return f"""
    <div style="margin: 10px 0; text-align: {align};">
        <div style="display: inline-block; max-width: 80%; text-align: left;">
            <div style="font-size: 0.8em; color: #666; margin-bottom: 3px;">
                <b>{label}</b> • {timestamp}
            </div>
            <div style="background-color: {bg_color}; padding: 12px 16px; border-radius: 8px; white-space: pre-wrap;">
                {content}
            </div>
        </div>
    </div>
    """

def process_query(b):
    """Handle user query when button is clicked."""
    query = input_box.value.strip()
    
    if not query:
        return
    
    # Clear input
    input_box.value = ""
    
    # Add user message to history
    timestamp = datetime.datetime.now().strftime("%H:%M")
    chat_history.append({"role": "user", "content": query, "timestamp": timestamp})
    
    # Update display with user message
    update_chat_display()
    
    # Show loading indicator
    with output:
        clear_output(wait=True)
        for msg in chat_history:
            display(HTML(format_message(msg["role"], msg["content"], msg["timestamp"])))
        display(HTML('<div style="margin: 10px 0;"><i>⏳ Thinking...</i></div>'))
    
    # Run RAG pipeline
    result = rag_pipeline(query, top_k=5, verbose=False)
    answer = result['answer']
    
    # Add source citations
    if result['sources']:
        answer += "\n\n---\n📚 Sources:\n"
        for i, source in enumerate(result['sources'][:3], 1):
            answer += f"\n{i}. {source['id']} (relevance: {source['score']:.2f})"
    
    # Add assistant response to history
    timestamp = datetime.datetime.now().strftime("%H:%M")
    chat_history.append({"role": "assistant", "content": answer, "timestamp": timestamp})
    
    # Update display
    update_chat_display()

def update_chat_display():
    """Update the chat display with current history."""
    with output:
        clear_output(wait=True)
        for msg in chat_history:
            display(HTML(format_message(msg["role"], msg["content"], msg["timestamp"])))

def clear_chat(b):
    """Clear chat history."""
    global chat_history
    chat_history = []
    with output:
        clear_output()
        display(HTML('<div style="padding: 20px; text-align: center; color: #999;"><i>Chat cleared. Ask me anything about NSW EV infrastructure!</i></div>'))

def try_example(b):
    """Load an example query."""
    input_box.value = b.description

# Create UI components
print("🚗⚡ NSW EV Intelligence Assistant")
print("="*80)

# Title
display(HTML("""
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 10px; margin-bottom: 20px;">
    <h2 style="margin: 0 0 10px 0;">🚗⚡ NSW EV Intelligence Assistant</h2>
    <p style="margin: 0; opacity: 0.9;">Ask me about EV charging stations, route safety, and trip planning in NSW</p>
</div>
"""))

# Example buttons
example_queries = [
    "Where can I find fast charging stations near Sydney CBD?",
    "What's the charging speed at Tesla supercharger stations?",
    "Are there any routes with safety hazards or delays?",
    "Find me charging stations operated by NRMA in regional NSW"
]

display(HTML('<div style="margin-bottom: 10px;"><b>Try these examples:</b></div>'))
example_buttons = [widgets.Button(description=q, layout=widgets.Layout(width='auto', margin='2px')) for q in example_queries]
for btn in example_buttons:
    btn.on_click(try_example)
    display(btn)

# Chat output area
output = widgets.Output(layout=widgets.Layout(height='400px', overflow_y='auto', border='1px solid #ddd', padding='10px', margin='10px 0'))
display(output)

with output:
    display(HTML('<div style="padding: 20px; text-align: center; color: #999;"><i>👋 Hello! Ask me anything about NSW EV infrastructure.</i></div>'))

# Input area
input_box = widgets.Text(
    placeholder='Type your question here...',
    layout=widgets.Layout(width='80%')
)

send_button = widgets.Button(
    description='Send',
    button_style='primary',
    layout=widgets.Layout(width='10%')
)

clear_button = widgets.Button(
    description='Clear',
    button_style='',
    layout=widgets.Layout(width='10%')
)

send_button.on_click(process_query)
clear_button.on_click(clear_chat)

# Allow Enter key to send
def handle_submit(sender):
    process_query(None)

input_box.on_submit(handle_submit)

# Display input controls
input_row = widgets.HBox([input_box, send_button, clear_button])
display(input_row)

print("\n✓ Chat interface ready! Type your question above or click an example.")

In [0]:
# The chat interface is already running in the cell above!
# No need to launch separately - ipywidgets displays inline.

print("✓ Chat interface is active!")
print("\nThe interactive chat UI is displayed above.")
print("You can:")
print("  • Type questions in the input box")
print("  • Click example buttons to try sample queries")
print("  • Press Enter or click Send to submit")
print("  • Click Clear to reset the conversation")
print("\nPowered by:")
print(f"  • Vector Search Index: {INDEX_NAME}")
print(f"  • LLM: {LLM_ENDPOINT}")
print(f"  • Documents: 117 EV stations + route safety reports")

## ✅ RAG Chat Application Complete!

You now have a fully functional RAG-powered chat application with:

1. ✓ **Retrieval Function** - Semantic search across your EV and route data
2. ✓ **LLM Integration** - Databricks DBRX Instruct for response generation
3. ✓ **Prompt Template** - Structured prompts that combine context + question
4. ✓ **Chat Interface** - Interactive Gradio UI for user interaction

## How It Works

```
User asks question
    ↓
Vector Search finds 5 most relevant documents
    ↓
Prompt combines documents + question
    ↓
LLM generates answer based on context
    ↓
User sees answer + source citations
```

## Enhancements You Can Add

### 1. **Deploy as Databricks App**
```python
# Save this notebook and reference it in your app.yaml
# The Gradio interface can run as a persistent web app
```

### 2. **Add Conversation Memory**
```python
# Store chat history in a session state
# Allow follow-up questions that reference previous context
```

### 3. **Filter by Location**
```python
# Add location input to filter results by region/postcode
# Use filters_json parameter in query_index()
```

### 4. **Multi-Modal Responses**
```python
# Generate maps showing charging station locations
# Create visualizations of route safety data
```

### 5. **Streaming Responses**
```python
# Stream LLM responses token-by-token for better UX
# Use Databricks streaming endpoints
```

## Testing Your RAG Application

Try these sample questions:
- "Where can I find fast charging stations near Sydney CBD?"
- "What's the charging speed at Tesla supercharger stations?"
- "Are there any routes with safety hazards or delays?"
- "Find me charging stations operated by NRMA in regional NSW"
- "What areas have the most EV charging infrastructure?"

## Production Considerations

1. **Error Handling** - Add retry logic for API failures
2. **Rate Limiting** - Implement request throttling
3. **Logging** - Track queries, response times, and user feedback
4. **Caching** - Cache common queries to reduce costs
5. **Monitoring** - Set up alerts for endpoint health

## Integrating with Your Existing App

Your app is defined in: `/NSW-EV-Intelligence-Platform/nsw-ev-intelligence/app.py`

You can:
- Import the `rag_pipeline` function into your app
- Replace the Gradio UI with your existing Streamlit UI
- Add RAG chat as a new tab/feature in your dashboard